In [1]:
%load_ext autoreload
%autoreload 2
%pip install websockets cryptography psutil aiohttp pywin32

from pathlib import Path
import discovery
import runtime_logging
import db_manager
import server_core
from server_core import handle_client, global_timer
# MERT'İN MODÜLÜNÜ İÇERİ AKTARIYORUZ
import db_manager

import asyncio
import websockets




Note: you may need to restart the kernel to use updated packages.


3) SUNUCUYU BAŞLATMA VE AÇIK TUTMA

In [2]:
print("🌐 [SUNUCU] Merkezi Sınav Sunucusu Başlatılıyor...")

# 1. Ahmet'in Sistem Hata Kaydedicisini Başlat
# capture_stdout=False ekleyerek Ahmet'in sistemiyle Notebook'un çakışmasını engelliyoruz
runtime_logging.setup_runtime_logging(
    process_name="EXAM_SERVER", 
    log_dir=Path("logs"), 
    capture_stdout=False  # Bunu ekle
)

import os # En üste eklemeyi unutma

async def run_server():
    # --- AYAR: TAM SIFIRLAMA ---
    HARD_RESET = True 

    if HARD_RESET:
        # YENİ EKLENDİ (ENGİN FIX): Windows dosya kilidini kaldırmak için DB bağlantısını kapat!
        # db_manager.py içinde close_db() fonksiyonu yoksa pass geçmemesi için try-except eklendi
        try:
            db_manager.close_db()
        except AttributeError:
            pass # Eğer Mert close_db fonksiyonunu henüz eklemediyse hata vermesin
            
        # 1. Eski SQLite veritabanı dosyasını güvenle sil
        if os.path.exists("exam_monitor.db"):
            try:
                os.remove("exam_monitor.db")
            except Exception as e:
                print(f"⚠️ [UYARI] DB dosyası silinemedi (Muhtemelen hala açık): {e}")
            
        # (Opsiyonel) Log dosyasını da temizle
        if os.path.exists("sinav_raporu_merkezi.jsonl"):
            try:
                os.remove("sinav_raporu_merkezi.jsonl")
            except Exception:
                pass
        
        # 2. RAM'deki listeleri tamamen boşalt
        server_core.active_students.clear()
        server_core.exam_registry.clear()
        
        # 3. Veritabanı tablolarını tertemiz dosyayla sıfırdan yeniden oluştur
        db_manager.init_db() 
        print("💥 [SİSTEM] Fabrika ayarlarına dönüldü. Veritabanı ve RAM SIFIRLANDI!")
    else:
        # ÇÖKME KURTARMA (CRASH RECOVERY) SİSTEMİ
        kurtarilan_durum = db_manager.load_server_state()
        if kurtarilan_durum:
            server_core.active_students = kurtarilan_durum.get("active_students", {})
            server_core.exam_registry = kurtarilan_durum.get("exam_registry", {})
            print(f"🔄 [RECOVERY] Sistem çökmeden kurtarıldı: {len(server_core.active_students)} öğrenci yüklendi.")
        else:
            server_core.active_students.clear()
            server_core.exam_registry.clear()
            print("🧹 [SİSTEM] Yedek bulunamadı, temiz başlanıyor.")

    server_core.dashboard_counter = 0
    
    # 3. Arka plan görevlerini başlat
    asyncio.create_task(global_timer())
    announcer = discovery.ServerAnnouncer(server_host="127.0.0.1", server_port=8765)
    await announcer.start()

    # Sunucuyu başlat
    async with websockets.serve(handle_client, "0.0.0.0", 8765):
        print("🌐 [SUNUCU] Port 8765 dinleniyor. İstemciler bekleniyor...\n")
        try:
            await asyncio.Future()  
        except asyncio.CancelledError:
            print("🛑 [SİSTEM] Sunucu durduruldu.")
        finally:
            await announcer.stop()

# Sistemi Başlat
await run_server()

🌐 [SUNUCU] Merkezi Sınav Sunucusu Başlatılıyor...
💥 [SİSTEM] Fabrika ayarlarına dönüldü. Veritabanı ve RAM SIFIRLANDI!
[DISCOVERY] Announcing 'default' on UDP port 5354 every 3.0s


[LOG] Writing runtime log to logs\EXAM_SERVER_20260417_124628.jsonl


🌐 [SUNUCU] Port 8765 dinleniyor. İstemciler bekleniyor...

📊 [SİSTEM] Dashboard güncellendi. (İstek: 5)
🚨 [ALARM] std_01 ihlal yaptı! Sınav donduruldu.
   ↳ 🧠 [ANALİZ] Güvenlik Skoru: %0 - Seviye: DÜŞÜK
   ↳ Sebep: FOCUS_LOST | Pencere: ● egitmen_dashboard.ipynb - SOFTWARE ENGİNEERİNG PROJECT - NETWORK-BASED LABORATORY EXAM SYSTEM - Visual Studio Code - Untracked
   ↳ Sıra No: 1 | Oluşturulma: 2026-04-17T12:46:53.375
📊 [SİSTEM] Dashboard güncellendi. (İstek: 7)
🚨 [ALARM] std_01 ihlal yaptı! Sınav donduruldu.
   ↳ 🧠 [ANALİZ] Güvenlik Skoru: %0 - Seviye: DÜŞÜK
   ↳ Sebep: FOCUS_LOST | Pencere: ● egitmen_dashboard.ipynb - SOFTWARE ENGİNEERİNG PROJECT - NETWORK-BASED LABORATORY EXAM SYSTEM - Visual Studio Code - Untracked
   ↳ Sıra No: 2 | Oluşturulma: 2026-04-17T12:46:58.388
📊 [SİSTEM] Dashboard güncellendi. (İstek: 10)
🚨 [ALARM] std_01 ihlal yaptı! Sınav donduruldu.
   ↳ 🧠 [ANALİZ] Güvenlik Skoru: %40 - Seviye: YÜKSEK
   ↳ Sebep: FOCUS_LOST | Pencere: WhatsApp
   ↳ Sıra No: 3 | Oluşturul